# Gene-set enrichment analysis(GSEA) Pipeline

**Author:** Yong Gyu Kim </br>
**Last Updated:** 2025-10-16 </br>
**Contact:** parkgilbong@gmail.com </br>

## Pipeline Overview

This Jupyter Notebook orchestrates a comprehensive RNA-Seq post-analysis workflow. It is designed to take raw differential expression results (e.g., from DESeq2) and perform **Gene Set Enrichment Analysis (GSEA)** in a systematic, reproducible, and highly automated manner.

The pipeline is built with modularity and reproducibility as its core principles. Each major analysis step is handled by a dedicated Python script, and the entire workflow is controlled by a single, centralized YAML configuration file.

### Key Features ✨

* **Dynamic & Reproducible Runs:** At the start of each run, a unique, timestamped output folder is created (e.g., `output/GSEA_Pipeline_2025-10-16_13-57-00/`). All results, logs, and the exact configuration file used for that specific run are saved together in this folder, ensuring perfect reproducibility.
* **Integrated Logging:** All output from the notebook and every script it calls is captured in a single, consolidated log file (`pipeline_run.log`) located within the unique output folder. This makes tracking the process and debugging errors incredibly simple.
* **Centralized Configuration:** All parameters for every step—from input file paths to statistical cutoffs—are managed in a single YAML configuration file. This allows for easy adaptation of the pipeline to different datasets and experiments without ever touching the code.
* **Comprehensive GSEA Screening:** The pipeline automatically runs GSEA against a user-defined list of MSigDB gene set collections (e.g., Hallmark, KEGG, GO) and custom GMT files.
* **Automated Reporting:** Final GSEA screening results are compiled into a single, multi-sheet Excel file, and corresponding summary plots (dot plots, bar plots) are generated for each gene set collection, facilitating rapid interpretation of the results.

---

## How to Use This Pipeline 🚀

### Step 1: Prepare Your Configuration File

1.  Choose a base configuration file from the `configs/` directory (e.g., `GSEA_Pipeline.yaml`) that matches your project.
2.  Open the file and carefully edit the parameters for each section. Pay close attention to:
    * **`data_loading`**: Specify the path to your raw data Excel file and the names of the required columns (`gene_col`, `log2fc_col`, etc.). If you plan to run Classic GSEA, also define the `gsea_outputs` section with your sample columns.
    * **`gsea`**: Define the `mode` ('prerank' or 'classic'). For screening, provide a list of gene sets under `gene_sets`. These can be official `gseapy` library names or relative paths to your own `.gmt` files in the `ref/` folder.
    * **`gsea_plot`**: Adjust the `fdr_cutoff` and `top_n_plots` to control how the final summary plots are generated.

### Step 2: Set the Base Configuration in the Notebook

### Step 3: Run the Notebook Cells Sequentially
* Execute the cells one by one. 
    - Cell 1 (Pipeline Initialization): This is the most important step. It will:
        * Create the unique output folder for this run.
        * Set up the integrated logging system.
        * Generate the final run_config.yaml that will be used by all subsequent steps.

    - Subsequent Cells: Each cell calls a specific analysis script (data_loading.py, gsea_analysis.py, etc.), passing it the path to the run_config.yaml file. Follow the markdown instructions for each cell.

### Step 4: Find Your Results
* Once the notebook has finished running, navigate to the unique output folder created in output/. Inside, you will find all your results neatly organized, including:
    - `run_config.yaml`: The exact settings used for this analysis.
    - `pipeline_run.log`: A complete log of everything that happened.
    - `standardized.csv`: The processed input data.
    - `GSEA_Report.xlsx`: The final multi-sheet GSEA screening results.
    - `gsea_plots/`: A folder containing all the generated dot plots and bar plots.
    - And all other intermediate files and detailed GSEA result folders.

---

## 1. Pipeline Initialization
These first cells are the control center for your entire analysis run. 🚀

Before you run it, please double-check that the `BASE_CONFIG_PATH` variable points to the correct base YAML configuration file you want to use for this analysis.

When you execute this cell, it will automatically:

1. Create a unique, timestamped output folder inside the output/ directory. This ensures that every run is saved separately and nothing is ever overwritten.

2. Set up the integrated logging system. A pipeline_run.log file will be created inside the new output folder. All messages from this notebook and every script it calls will be saved to this single file.

3. Generate the final run_config.yaml file for this specific run and save it inside the output folder. This file contains the exact settings used, ensuring your analysis is perfectly reproducible.

Simply run the cell below to get started

In [1]:
import os
import yaml
import logging
from datetime import datetime
from pathlib import Path

from IPython.display import Image, display
from utils.logging_utils_environ import setup_logging

# --- 1. Define Base Configuration ---
BASE_CONFIG_PATH = Path("../configs/GSEA_pipeline.yaml") # Change this to your desired base config file
PIPELINE_NAME = BASE_CONFIG_PATH.stem

# --- 2. Create a Unique Timestamped Output Directory for this Run ---
# This folder will contain EVERYTHING for this run: results, logs, and the config.
PROJECT_ROOT = Path.cwd().parent
print(f"Project root = {PROJECT_ROOT}")
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_output_dir = PROJECT_ROOT / f"output/{PIPELINE_NAME}_{timestamp}"
run_output_dir.mkdir(parents=True, exist_ok=True)

# --- 3. Set Up Logging to Save Inside the Unique Output Directory ---
log_file_path = run_output_dir / "pipeline_run.log"
os.environ['PIPELINE_LOG_FILE'] = str(log_file_path)
logger, _ = setup_logging()

# --- 4. Create the Final, Dynamic Configuration for this Run ---
# Read the base settings
with open(BASE_CONFIG_PATH, 'r', encoding='utf-8') as f:
    config_data = yaml.safe_load(f)

# Dynamically set the 'ROOT_DIR' to our unique output folder
config_data['ROOT_DIR'] = str(run_output_dir)

# Define the final path for the config file for THIS RUN.
# No more temp files!
RUN_CONFIG_PATH = run_output_dir / "run_config.yaml"

# Save the final, complete config file directly into the output directory.
with open(RUN_CONFIG_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(config_data, f, default_flow_style=False, sort_keys=False)

# --- 5. Final Confirmation ---
logger.info(f"✅ Pipeline '{PIPELINE_NAME}' run initialized.")
logger.info(f"📂 All results, logs, and the exact config used will be saved in:\n{run_output_dir.resolve()}")

# This CONFIG_PATH variable is now the single source of truth for this run.
# It will be used by all subsequent cells in the notebook.
CONFIG_PATH = str(RUN_CONFIG_PATH)

2025-10-16 15:37:13,866 | MyAnalysisLogger | INFO | Attached to pipeline log file: output\GSEA_pipeline_2025-10-16_15-37-13\pipeline_run.log
2025-10-16 15:37:13,896 | MyAnalysisLogger | INFO | ✅ Pipeline 'GSEA_pipeline' run initialized.
2025-10-16 15:37:13,902 | MyAnalysisLogger | INFO | 📂 All results, logs, and the exact config used will be saved in:
C:\Users\KimYG\Documents\GitHub\RNA-Seq_GO_analysis\notebooks\output\GSEA_pipeline_2025-10-16_15-37-13


In [2]:
# Environment setup (paths, PYTHONPATH, folders)
# 1. 프로젝트의 최상위 경로(PROJECT_ROOT)를 정의합니다.
# 현재 노트북 파일이 있는 폴더를 기준으로 경로를 설정합니다.
# PROJECT_ROOT = Path.cwd().parent  # NOTE: 하위 폴더에서 실행 시 적절히 조정

# 2. 실행할 스크립트와 설정 파일의 절대 경로를 동적으로 생성합니다.
CONFIG_DIR = PROJECT_ROOT / "configs"
RESULTS_DIR = PROJECT_ROOT / "results"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# CONFIG_PATH = BASE_CONFIG_PATH
SRC_DIR = PROJECT_ROOT / "src" / "analysis"

data_loading_PATH = SRC_DIR / "data_loading.py"
gsea_PATH = SRC_DIR / "gsea_analysis.py"
gsea_plot_PATH = SRC_DIR / "gsea_plot.py"
report_PATH = SRC_DIR / "report_generation.py"

for d in (CONFIG_DIR, RESULTS_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

logger.info(f"Project root = {PROJECT_ROOT}")
logger.info(f"Config path = {CONFIG_PATH}")

2025-10-16 15:37:17,389 | MyAnalysisLogger | INFO | Project root = c:\Users\KimYG\Documents\GitHub\RNA-Seq_GO_analysis
2025-10-16 15:37:17,390 | MyAnalysisLogger | INFO | Config path = output\GSEA_pipeline_2025-10-16_15-37-13\run_config.yaml


## 2. Standardize data (Excel -> CSV)
This cell executes the data loading and preparation step. 🧹

It runs the `data_loading.py` script, which uses the `run_config.yaml` file created in the previous step. The script will:

* Read your raw data from the Excel file specified in the config.

* Standardize the column names to `gene`, `log2fc`, and `padj` while keeping all other original columns.

* Save the processed data as `standardized.csv` inside your unique output folder.

* If you enabled gsea_outputs in your config, it will also generate the `gsea_expression_matrix.txt` and `gsea_class_labels.cls` files required for Classic GSEA.

Run the cell below to process your input data.

In [3]:
command_dl = f"python {data_loading_PATH} --config {CONFIG_PATH} --config-section data_loading"
logger.info(f"Executing command: \n{command_dl}")
!{command_dl}

2025-10-16 15:37:22,075 | MyAnalysisLogger | INFO | Executing command: 
python c:\Users\KimYG\Documents\GitHub\RNA-Seq_GO_analysis\src\analysis\data_loading.py --config output\GSEA_pipeline_2025-10-16_15-37-13\run_config.yaml --config-section data_loading
2025-10-16 15:37:23,829 | MyAnalysisLogger | INFO | Attached to pipeline log file: output\GSEA_pipeline_2025-10-16_15-37-13\pipeline_run.log
2025-10-16 15:37:31,754 | MyAnalysisLogger | INFO | Loaded rows=14237 from files=1
2025-10-16 15:37:32,151 | MyAnalysisLogger | INFO | Saved standardized data to: C:\Users\KimYG\Documents\GitHub\RNA-Seq_GO_analysis\output\GSEA_pipeline_2025-10-16_15-37-13\standardized.csv
2025-10-16 15:37:32,151 | MyAnalysisLogger | INFO | GSEA output generation is enabled.
2025-10-16 15:37:32,378 | MyAnalysisLogger | INFO | Successfully saved GSEA expression matrix to: C:\Users\KimYG\Documents\GitHub\RNA-Seq_GO_analysis\output\GSEA_pipeline_2025-10-16_15-37-13\gsea_expression_matrix.txt
2025-10-16 15:37:32,383 |

In [4]:
import pandas as pd
from pathlib import Path as _P

gsea_expression_matrix = f"../{run_output_dir}/gsea_expression_matrix.txt"
if _P(gsea_expression_matrix).exists():
    df_std = pd.read_table(gsea_expression_matrix)
    display(df_std.head())
    print("rows:", len(df_std))
else:
    print("Missing", gsea_expression_matrix, "- check your config paths.")

,NAME,DESCRIPTION,HT1,HT2,HT3,HT4,HT5,WT1,WT2,WT3,WT4,WT5
0,Shank2,Shank2,6167.020884,6970.171866,6503.145544,6756.092005,7139.781890,9356.939549,9144.554630,8583.970218,9354.286839,10950.033061
1,Mdp1,Mdp1,847.921190,795.825358,804.353733,784.550125,796.432588,878.234222,962.269843,936.598108,1001.725429,960.577350
2,Cp,Cp,548.670190,488.395429,446.249349,382.742724,457.793586,361.647239,320.888916,346.237629,294.959900,352.496773
3,4931414P19Rik,4931414P19Rik,158.660565,169.790417,118.934073,126.769365,132.514145,62.507250,86.306754,78.027874,97.502948,70.947736
4,Cep350,Cep350,1868.897537,2249.094444,2359.417652,2122.690930,2141.156104,1816.324404,1712.700316,1717.873960,1833.646575,1659.744362


rows: 14237


## 3. Gene-set enrichment analysis (GSEA)
This is the main analysis step where the GSEA screening is performed. 🔬

This cell runs the gsea_analysis.py script. Based on your run_config.yaml settings, it will:

* Run in either 'prerank' or 'classic' mode.

* Iterate through the entire list of gene sets you provided in the gene_sets section of your config.

* For each gene set collection, it will run a full GSEA analysis and save the detailed results (including individual enrichment plots) in a dedicated sub-folder.

* Finally, it will compile the summary statistics from all runs into a single, multi-sheet GSEA_Report.xlsx file inside your output folder.

This step may take some time depending on the number of gene sets and the permutation_num you've set.

In [7]:
command_gsea = f"python {gsea_PATH} --config {CONFIG_PATH} --config-section gsea"
logger.info(f"Executing command: \n{command_gsea}")
!{command_gsea}

2025-10-16 15:46:24,963 | MyAnalysisLogger | INFO | Executing command: 
python c:\Users\KimYG\Documents\GitHub\RNA-Seq_GO_analysis\src\analysis\gsea_analysis.py --config output\GSEA_pipeline_2025-10-16_15-37-13\run_config.yaml --config-section gsea
2025-10-16 15:46:29,031 | MyAnalysisLogger | INFO | Attached to pipeline log file: output\GSEA_pipeline_2025-10-16_15-37-13\pipeline_run.log

2025-10-16 15:46:29,950 [INFO] Parsing data files for GSEA.............................
2025-10-16 15:46:30,018 [INFO] Enrichr library gene sets already downloaded in: C:\Users\KimYG\.cache/gseapy, use local file
2025-10-16 15:46:30,052 [INFO] 0030 gene_sets have been filtered out when max_size=500 and min_size=15
2025-10-16 15:46:30,053 [INFO] 0290 gene_sets used for further statistical testing.....
2025-10-16 15:46:30,053 [INFO] Start to run GSEA...Might take a while..................
2025-10-16 15:46:30,057 [INFO] Genes are converted to uppercase.
2025-10-16 15:46:45,218 [INFO] Start to generate GSEApy reports and figures............
2025-10-16 15:46:58,566 [INFO] Congratulations. GSEApy ran successfully.................

2025-10-16 15:46:58,626 [INFO] Parsing data files for GSEA.............................
2025-10-16 15:46:58,691 [INFO] Enrichr library gene sets already downloaded in: C:\Users\KimYG\.cache/gseapy, use local file
2025-10-16 15:46:58,844 [INFO] 3116 gene_sets have b


2025-10-16 15:46:29,040 | MyAnalysisLogger | INFO | GSEA 스크리닝 모드: 'classic'
2025-10-16 15:46:29,040 | MyAnalysisLogger | INFO | GSEA 분석 파라미터: {'min_size': 15, 'max_size': 500, 'permutation_num': 1000, 'weighting_exponent': 1, 'seed': 42}
2025-10-16 15:46:29,946 | MyAnalysisLogger | INFO | 총 4개의 유전자 세트에 대한 GSEA 스크리닝을 시작합니다.
2025-10-16 15:46:29,947 | MyAnalysisLogger | INFO | 프로젝트 루트 경로를 'C:\Users\KimYG\Documents\GitHub\RNA-Seq_GO_analysis'로 설정합니다.
2025-10-16 15:46:29,947 | MyAnalysisLogger | INFO | --- Classic GSEA 분석 시작: 'KEGG_2021_Human' ---
2025-10-16 15:46:58,620 | MyAnalysisLogger | INFO | --- Classic GSEA 분석 시작: 'GO_Biological_Process_2023' ---
2025-10-16 15:49:06,877 | MyAnalysisLogger | INFO | --- Classic GSEA 분석 시작: 'MSigDB_Hallmark_2020' ---
2025-10-16 15:49:30,762 | MyAnalysisLogger | INFO | --- Classic GSEA 분석 시작: 'mh.all.v2025.1.Mm.symbols.gmt' ---
2025-10-16 15:49:54,119 | MyAnalysisLogger | INFO | 모든 GSEA 결과를 하나의 Excel 파일로 통합 중: C:\Users\KimYG\Documents\GitHub\RNA-Seq_GO

In [8]:
gsea_result_csv = f"../{run_output_dir}/GSEA_Report.xlsx"
if _P(gsea_result_csv).exists():
    df_gsea = pd.read_excel(gsea_result_csv)
    display(df_gsea.head(10))
    print("rows:", len(df_gsea))
else:
    print("Missing", gsea_result_csv)

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,gsea,Ribosome,-0.704136,-1.897057,0.007984,0.036452,0.016,80/125,17.43%,RPL19;RPL13A;FAU;RPL9;RPL11;RPS27A;RPL4;RPL23A...
1,gsea,Glutathione metabolism,-0.539416,-1.862703,0.008230,0.026558,0.029,13/45,10.04%,GPX4;PGD;PRDX6;GSTM4;IDH2;GSTM5;SRM;MGST3;GSS;...
2,gsea,Coronavirus disease,-0.539624,-1.833488,0.008214,0.030551,0.053,68/171,13.94%,RPL19;RPL13A;FAU;RPL9;RPL11;RPS27A;RPL4;RPL23A...
3,gsea,Mineral absorption,0.467696,1.615297,0.008247,1.000000,0.504,7/36,4.04%,ATP2B4;ATP2B2;SLC8A1;TRPM6;ATP2B1;ATP2B3;SLC6A19
4,gsea,Nicotine addiction,0.560287,1.562529,0.060417,1.000000,0.658,11/36,7.48%,SLC17A6;GRIN2B;CHRNB2;GABRB2;CHRNA7;CACNA1B;GR...
5,gsea,Calcium signaling pathway,0.363865,1.553104,0.020704,0.882601,0.683,72/213,21.53%,ATP2B4;HTR4;ATP2B2;SLC8A1;CAMK1D;CHRM2;CACNA1E...
6,gsea,Endocrine and other factor-regulated calcium r...,0.501150,1.520677,0.061983,0.905512,0.801,14/45,10.30%,ATP2B4;ATP2B2;SLC8A1;KL;DNM3;ATP2B1;ATP2B3;GNA...
7,gsea,Salivary secretion,0.421543,1.520403,0.081081,0.725176,0.801,22/61,16.72%,ATP2B4;ATP2B2;GUCY1A2;ADCY1;ATP2B1;ATP2B3;GNAQ...
8,gsea,Circadian rhythm,0.544508,1.519573,0.037267,0.609902,0.801,11/28,15.55%,CLOCK;FBXW11;BTRC;PRKAA2;ARNTL;PER1;RORB;RORA;...
9,gsea,Long-term potentiation,0.426664,1.511605,0.030108,0.559175,0.819,19/62,10.30%,GRIN2B;CREBBP;GRM1;ADCY1;CAMK4;CACNA1C;GNAQ;PR...


rows: 290


## 4. GSEA plots
This final cell visualizes the results from your GSEA screening, making them easy to interpret. 📊

It runs the gsea_plot.py script, which will:

* Read the multi-sheet GSEA_Report.xlsx file.

* Go through every sheet in the Excel file one by one.

* For each sheet, it filters for statistically significant pathways using the fdr_cutoff from your config.

* It then generates a Dot Plot and a Bar Plot for the top significant pathways, controlled by top_n_plots.

* All generated plots are saved in the gsea_plots folder within your unique output directory.

Run the cell below to create your final summary figures.

In [9]:
command_gsea_plot = f"python {gsea_plot_PATH} --config {CONFIG_PATH} --config-section gsea_plot"
logger.info(f"Executing command: \n{command_gsea_plot}")
!{command_gsea_plot}

2025-10-16 15:50:29,048 | MyAnalysisLogger | INFO | Executing command: 
python c:\Users\KimYG\Documents\GitHub\RNA-Seq_GO_analysis\src\analysis\gsea_plot.py --config output\GSEA_pipeline_2025-10-16_15-37-13\run_config.yaml --config-section gsea_plot
2025-10-16 15:50:33,819 | MyAnalysisLogger | INFO | Attached to pipeline log file: output\GSEA_pipeline_2025-10-16_15-37-13\pipeline_run.log
2025-10-16 15:50:34,674 | MyAnalysisLogger | INFO | GSEA Excel 리포트로부터 플롯 생성을 시작합니다: GSEA_Report.xlsx
2025-10-16 15:50:35,216 | MyAnalysisLogger | INFO | --- 시트 처리 중: 'KEGG_2021_Human' ---
2025-10-16 15:50:35,363 | MyAnalysisLogger | INFO | 'KEGG_2021_Human' 시트에서 3개의 유의미한 유전자 세트를 찾았습니다. 상위 15개를 시각화합니다.
2025-10-16 15:50:36,548 | MyAnalysisLogger | INFO | Dot plot 생성 완료: dotplot_KEGG_2021_Human.png
2025-10-16 15:50:37,783 | MyAnalysisLogger | INFO | Bar plot 생성 완료: barplot_KEGG_2021_Human.png
2025-10-16 15:50:37,784 | MyAnalysisLogger | INFO | --- 시트 처리 중: 'GO_Biological_Process_2023' ---
2025-10-16 15:50